# LeetCode 691 — Stickers to Spell Word bằng Python

- Link: https://leetcode.com/problems/stickers-to-spell-word/
- Độ khó: **Hard**
- Chủ đề: Đếm ký tự (`Counter`), Quay lui có ghi nhớ (memoization), Quy hoạch động trên bitmask, BFS trên trạng thái

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minStickers(self, stickers: List[str], target: str) -> int:

```

## 1. Giới thiệu & mục tiêu học

Bạn có nhiều loại nhãn dán (sticker), mỗi nhãn là một từ; mỗi loại có **vô hạn** bản. Bạn được cắt rời từng chữ cái của nhãn để ghép thành từ `target`. Hỏi cần **ít nhất bao nhiêu nhãn**?

Đây là một bài "phủ tập hợp có lặp" (set cover với multiset) — tổng quát thì là NP-khó, nhưng `target` dài **tối đa 15** nên ta có thể mã hoá "những vị trí nào của `target` đã có chữ" bằng **bitmask 15 bit** (`2^15 = 32768` trạng thái). Bài này là bài tập tuyệt vời để học cách **thiết kế trạng thái** cho DP/BFS và các mẹo cắt tỉa.

Sau notebook này bạn sẽ:

1. Biểu diễn từ dưới dạng **đếm ký tự** (`Counter`, mảng 26 ô) và biết "dùng một nhãn" nghĩa là trừ bớt các chữ cần.
2. Viết brute force "thử số nhãn `t = 1, 2, …`" để làm trọng tài.
3. Hiểu mẹo cắt tỉa then chốt: **chỉ thử các nhãn chứa chữ cái đầu tiên còn thiếu**.
4. Viết DFS có ghi nhớ trên chuỗi còn lại, DP trên bitmask và BFS trên bitmask.
5. Kiểm chứng mọi cách bằng test ngẫu nhiên có seed; biết vì sao tham lam "chọn nhãn phủ nhiều nhất" là sai.

## 2. Đề bài & ví dụ

**Phát biểu lại.** Cho `n` loại nhãn dán `stickers`, mỗi nhãn là một chuỗi chữ thường. Bạn muốn ghép thành chuỗi `target` bằng cách cắt từng chữ cái từ các nhãn đã mua rồi sắp xếp lại. Mỗi loại nhãn có thể dùng **nhiều lần** (vô hạn bản). Trả về **số nhãn ít nhất** cần dùng; nếu không thể, trả về `-1`.

Lưu ý: mỗi chữ cái trên một nhãn chỉ dùng được một lần, chữ thừa thì bỏ.

**Ví dụ 1**

```text
Input:  stickers = ["with","example","science"], target = "thehat"
Output: 3
Giải thích: dùng 2 nhãn "with" và 1 nhãn "example":
  "with" + "with" cho t, h, t, h (và w, i, w, i thừa); "example" cho e, a.
  Ghép được t-h-e-h-a-t. Không thể làm với ít hơn 3 nhãn.
```

**Ví dụ 2**

```text
Input:  stickers = ["notice","possible"], target = "basicbasic"
Output: -1
Giải thích: không nhãn nào có chữ 'a' → không thể ghép.
```

**Ràng buộc**

- `n == stickers.length`, `1 <= n <= 50`
- `1 <= stickers[i].length <= 10`
- `1 <= target.length <= 15`
- `stickers[i]` và `target` gồm chữ cái tiếng Anh thường.

In [1]:
from collections import Counter

st1, tg1 = ["with", "example", "science"], "thehat"
st2, tg2 = ["notice", "possible"], "basicbasic"

print("cần:", Counter(tg1))
print("2×with + example cho:", Counter("with") + Counter("with") + Counter("example"))
have = Counter("with") + Counter("with") + Counter("example")
print("đủ chưa?", all(have[c] >= v for c, v in Counter(tg1).items()))
print("chữ trong target mà không nhãn nào có (ví dụ 2):", set(tg2) - set(''.join(st2)))

cần: Counter({'t': 2, 'h': 2, 'e': 1, 'a': 1})
2×with + example cho: Counter({'w': 2, 'i': 2, 't': 2, 'h': 2, 'e': 2, 'x': 1, 'a': 1, 'm': 1, 'p': 1, 'l': 1})
đủ chưa? True
chữ trong target mà không nhãn nào có (ví dụ 2): {'a'}


## 3. Kiến thức nền

### 3.1. `Counter` — đếm ký tự

`Counter("hello")` cho `{'l': 2, 'h': 1, 'e': 1, 'o': 1}`. Các phép hay dùng:

- `c1 + c2`: cộng số lượng.
- `c1 - c2`: trừ, **bỏ các số ≤ 0**.
- `c1 & c2`: lấy **min** từng chữ (giao).

In [2]:
need = Counter("thehat")
st = Counter("with")
print("need       :", need)
print("need - with:", need - st)          # sau khi dùng 1 nhãn "with"
print("need & with:", need & st)          # những chữ của "with" thật sự có ích

need       : Counter({'t': 2, 'h': 2, 'e': 1, 'a': 1})
need - with: Counter({'t': 1, 'h': 1, 'e': 1, 'a': 1})
need & with: Counter({'t': 1, 'h': 1})


### 3.2. Mảng 26 ô thay cho `Counter` (nhanh hơn)

`ord(ch) - ord('a')` đổi chữ cái thành số 0..25. Một từ có thể biểu diễn bằng list 26 số đếm.

In [3]:
def to_count(word):
    cnt = [0] * 26
    for ch in word:
        cnt[ord(ch) - 97] += 1
    return cnt

c = to_count("example")
print({chr(97 + i): v for i, v in enumerate(c) if v})

{'a': 1, 'e': 2, 'l': 1, 'm': 1, 'p': 1, 'x': 1}


### 3.3. Bitmask trên vị trí của `target`

Bit `j` của `mask` bật ⇔ vị trí `j` của `target` **đã có chữ**. `mask = 0` là chưa có gì, `mask = 2^L - 1` là đã đủ.

- Kiểm tra bit: `mask >> j & 1`.
- Bật bit: `mask | (1 << j)`.

In [4]:
target = "thehat"
L = len(target)
mask = 0
mask |= 1 << 0          # có 't' ở vị trí 0
mask |= 1 << 5          # có 't' ở vị trí 5
print(format(mask, f'0{L}b')[::-1], "(đọc trái→phải là vị trí 0..5)")
print("vị trí còn thiếu:", [j for j in range(L) if not mask >> j & 1])
print("đủ chưa?", mask == (1 << L) - 1)

100001 (đọc trái→phải là vị trí 0..5)
vị trí còn thiếu: [1, 2, 3, 4]
đủ chưa? False


### 3.4. `functools.lru_cache` — ghi nhớ kết quả hàm

Gắn `@lru_cache(maxsize=None)` lên hàm đệ quy, Python tự lưu kết quả theo tham số. Tham số phải **hashable** (chuỗi, tuple, số — không phải list).

In [5]:
from functools import lru_cache

@lru_cache(maxsize=None)
def fib(n):
    return n if n < 2 else fib(n - 1) + fib(n - 2)

print(fib(80), fib.cache_info())

23416728348467685 CacheInfo(hits=78, misses=81, maxsize=None, currsize=81)


### 3.5. `combinations_with_replacement` — chọn có lặp, không thứ tự

Chọn `t` nhãn trong `u` loại, được lặp, thứ tự không quan trọng: `combinations_with_replacement(range(u), t)`.

In [6]:
from itertools import combinations_with_replacement
print(list(combinations_with_replacement(['with', 'example'], 2)))

[('with', 'with'), ('with', 'example'), ('example', 'example')]


## 4. Ý tưởng nền tảng / trực giác

### 4.1. Thứ tự dùng nhãn không quan trọng

Kết quả chỉ phụ thuộc **tập (có lặp)** các nhãn đã mua: tổng số chữ cái của chúng có đủ cho `target` không. Vậy một lời giải là một **multiset** nhãn.

### 4.2. Tính tay ví dụ 1

`target = "thehat"` cần: `t×2, h×2, e×1, a×1`.

| Nhãn | Chữ có ích (giao với cần) |
|---|---|
| `with` | t, h |
| `example` | e, a |
| `science` | e |

- 1 nhãn: tối đa 2 chữ có ích → không đủ 6.
- 2 nhãn: tối đa 4 chữ có ích → không đủ.
- 3 nhãn: `with + with + example` → t2 h2 e1 a1 ✓. Đáp án **3**.

### 4.3. "Còn thiếu gì" là trạng thái

Sau khi dùng vài nhãn, điều duy nhất quan trọng là **phần còn thiếu** của `target`. Ví dụ dùng `with` → còn thiếu `"eaht"` (thứ tự không quan trọng, nên ta có thể sắp xếp: `"aeht"`). Bài toán trở thành: từ trạng thái "còn thiếu S", cần ít nhất bao nhiêu nhãn? → hàm `f(S)`.

In [7]:
def use_sticker(remaining, sticker):
    """remaining: chuỗi đã sắp xếp; trả về chuỗi còn thiếu (đã sắp xếp) sau khi dùng 1 nhãn."""
    left = Counter(remaining) - Counter(sticker)
    return ''.join(sorted(left.elements()))

s = ''.join(sorted("thehat"))
print("ban đầu  :", s)
s = use_sticker(s, "with");    print("sau with :", s)
s = use_sticker(s, "with");    print("sau with :", s)
s = use_sticker(s, "example"); print("sau example:", repr(s), "-> xong!")

ban đầu  : aehhtt
sau with : aeht
sau with : ae
sau example: '' -> xong!


## 5. Cách 1 — Brute Force: thử `t = 1, 2, 3, …` nhãn

**Ý tưởng.**

1. Nếu `target` có chữ nào không nhãn nào có → `-1`.
2. Mỗi nhãn chỉ giữ phần **có ích** (`Counter(s) & need`); bỏ các nhãn vô ích.
3. Với `t = 1, 2, …, len(target)`: thử mọi multiset `t` nhãn; nếu tổng đủ → trả `t`. (Mỗi nhãn có ích góp ≥ 1 chữ cần, nên `t ≤ len(target)` luôn đủ nếu có lời giải.)

In [8]:
def stickers_brute(stickers, target):
    need = Counter(target)
    useful = [Counter(s) & need for s in stickers]
    useful = [c for c in useful if c]
    available = set()
    for c in useful:
        available |= set(c)
    if set(need) - available:
        return -1
    for t in range(1, len(target) + 1):
        for combo in combinations_with_replacement(range(len(useful)), t):
            total = Counter()
            for i in combo:
                total += useful[i]
            if all(total[ch] >= v for ch, v in need.items()):
                return t
    return -1

print(stickers_brute(st1, tg1))   # 3
print(stickers_brute(st2, tg2))   # -1

3
-1


**Độ phức tạp.** Số multiset cỡ `C(u + t - 1, t)`; với `u = 50`, `t = 15` là khoảng `10^14` — không thể. Chỉ dùng được khi `u` và `target` nhỏ.

**Nút thắt.**

- Ta xét rất nhiều multiset chỉ khác nhau ở nhãn "không cần thiết".
- Không tận dụng được việc nhiều cách khác nhau dẫn tới **cùng một phần còn thiếu**.

## 6. Bước đột phá

**Insight 1 — trạng thái nhỏ.** Phần còn thiếu luôn là một "tập con có lặp" của `target`. Có tối đa `2^15` cách chọn vị trí → ghi nhớ được. Viết `f(S)` = số nhãn ít nhất để phủ phần còn thiếu `S`:

> `f("") = 0`; `f(S) = 1 + min over nhãn s có ích của f(S - s)`.

**Insight 2 — cắt tỉa "chữ đầu tiên".** Trong mọi lời giải cho `S`, chữ cái `S[0]` (chữ đầu tiên còn thiếu) **phải** được phủ bởi một nhãn nào đó chứa nó. Vì thứ tự không quan trọng, ta có thể coi nhãn đó được dùng **trước tiên**. Vậy:

> **Nếu** nhãn `s` **không chứa** `S[0]` **thì** không cần thử `s` ở bước này.

Cắt tỉa này không làm mất lời giải tối ưu, nhưng giảm mạnh số nhánh: mỗi trạng thái chỉ thử các nhãn chứa một chữ cụ thể, và tránh việc thử cùng một multiset theo nhiều thứ tự khác nhau.

**Insight 3 — bitmask thay cho chuỗi.** Mã hoá phần đã có bằng bitmask vị trí. Dùng một nhãn = với mỗi chữ của nhãn, bật **vị trí chưa có đầu tiên** mang chữ đó. Các chuyển trạng thái `mask → mask'` luôn làm `mask` **tăng** → DP theo thứ tự tăng dần của `mask` là hợp lệ; hoặc BFS (mỗi cạnh tốn 1 nhãn).

**Bất biến:** mọi trạng thái đều mô tả chính xác multiset chữ còn thiếu; chuỗi `S` luôn được **sắp xếp** để hai cách đi khác nhau tới cùng multiset có chung một khoá memo.

## 7. Các cách tiếp theo

Tổng cộng 4 cách:

| Cách | Ý tưởng | Trạng thái |
|---|---|---|
| 1 | Brute force thử `t` nhãn | multiset nhãn |
| 2 | DFS + memo trên chuỗi còn thiếu (sắp xếp), cắt tỉa chữ đầu | chuỗi |
| 3 | DP trên bitmask vị trí, duyệt `mask` tăng dần | `2^L` mask |
| 4 | BFS trên bitmask (mỗi bước = 1 nhãn) | `2^L` mask |

### 7.1. Cách 2 — DFS có ghi nhớ trên chuỗi còn thiếu

In [9]:
def stickers_memo(stickers, target):
    counts = [to_count(s) for s in stickers]

    @lru_cache(maxsize=None)
    def f(rem):                       # rem: chuỗi còn thiếu, đã sắp xếp
        if not rem:
            return 0
        need = to_count(rem)
        first = ord(rem[0]) - 97
        best = float('inf')
        for cnt in counts:
            if cnt[first] == 0:       # cắt tỉa: nhãn phải chứa chữ đầu tiên còn thiếu
                continue
            nxt = []
            for i in range(26):
                if need[i] > cnt[i]:
                    nxt.append(chr(97 + i) * (need[i] - cnt[i]))
            best = min(best, 1 + f(''.join(nxt)))
        return best

    ans = f(''.join(sorted(target)))
    return -1 if ans == float('inf') else ans

print(stickers_memo(st1, tg1), stickers_memo(st2, tg2))

3 -1


Giải thích từng phần:

- `counts`: mỗi nhãn đổi thành mảng 26 số đếm một lần (đỡ tính lại).
- `rem` luôn sắp xếp: chuỗi `nxt` được ghép theo thứ tự chữ cái `a..z` nên tự động đã sắp xếp.
- `need[i] > cnt[i]`: chữ `i` còn thiếu `need[i] - cnt[i]` sau khi dùng nhãn.
- Nếu không nhãn nào chứa `rem[0]` → `best` giữ `inf` → lan truyền lên → trả `-1`.
- Mỗi lần gọi đệ quy, `rem` ngắn đi ít nhất 1 chữ (vì nhãn chứa `rem[0]`) → không lặp vô hạn.

**Code nộp LeetCode (Cách 2):**

```python
from functools import lru_cache

class Solution:
    def minStickers(self, stickers: List[str], target: str) -> int:
        counts = []
        for s in stickers:
            c = [0] * 26
            for ch in s:
                c[ord(ch) - 97] += 1
            counts.append(c)

        @lru_cache(maxsize=None)
        def f(rem):
            if not rem:
                return 0
            need = [0] * 26
            for ch in rem:
                need[ord(ch) - 97] += 1
            first = ord(rem[0]) - 97
            best = float('inf')
            for cnt in counts:
                if cnt[first] == 0:
                    continue
                nxt = ''.join(chr(97 + i) * (need[i] - cnt[i]) for i in range(26) if need[i] > cnt[i])
                best = min(best, 1 + f(nxt))
            return best

        ans = f(''.join(sorted(target)))
        return -1 if ans == float('inf') else ans
```

**Độ phức tạp.** Số trạng thái ≤ số multiset con của `target` ≤ `2^L`; mỗi trạng thái thử ≤ `n` nhãn, mỗi lần `O(26 + L)`. Tổng `O(2^L · n · (26 + L))` tệ nhất, nhưng thực tế ít hơn nhiều vì chữ trùng nhau gộp trạng thái và cắt tỉa chữ đầu.

### 7.2. Cách 3 — DP trên bitmask vị trí

**Giải thích.** `dp[mask]` = số nhãn ít nhất để đạt trạng thái "các vị trí trong `mask` đã có chữ". Khởi tạo `dp[0] = 0`. Duyệt `mask` **tăng dần**; nếu `dp[mask]` hữu hạn:

1. Tìm vị trí chưa có đầu tiên `j` (cắt tỉa "chữ đầu").
2. Với mỗi nhãn chứa `target[j]`: áp nhãn lên `mask` (mỗi chữ của nhãn bật vị trí chưa có **đầu tiên** cùng chữ) → `new`; `dp[new] = min(dp[new], dp[mask] + 1)`.

Vì `new ⊋ mask` nên `new > mask` (số học) → khi duyệt tới `new` thì `dp[new]` đã chốt.

In [10]:
def apply_sticker(mask, cnt, target):
    """Áp nhãn (mảng đếm cnt) lên mask: mỗi chữ của nhãn bật vị trí chưa có đầu tiên cùng chữ."""
    cnt = cnt[:]                               # copy để không sửa mảng gốc
    for j, ch in enumerate(target):
        if not mask >> j & 1:
            c = ord(ch) - 97
            if cnt[c] > 0:
                cnt[c] -= 1
                mask |= 1 << j
    return mask

def stickers_bitmask_dp(stickers, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    INF = float('inf')
    dp = [INF] * (1 << L)
    dp[0] = 0
    for mask in range(1 << L):
        if dp[mask] == INF:
            continue
        if mask == full:
            break
        j = 0
        while mask >> j & 1:                   # vị trí chưa có đầu tiên
            j += 1
        c = ord(target[j]) - 97
        for cnt in counts:
            if cnt[c] == 0:
                continue
            new = apply_sticker(mask, cnt, target)
            if dp[mask] + 1 < dp[new]:
                dp[new] = dp[mask] + 1
    return -1 if dp[full] == INF else dp[full]

print(stickers_bitmask_dp(st1, tg1), stickers_bitmask_dp(st2, tg2))

3 -1


**Vì sao "bật vị trí chưa có đầu tiên cùng chữ" không làm mất lời giải?** Các vị trí mang cùng một chữ là **hoán đổi được** cho nhau: việc chữ `'t'` lấp vị trí 0 hay vị trí 5 không ảnh hưởng tới phần còn lại. Ta luôn lấp theo một quy tắc cố định để không sinh ra các trạng thái "tương đương nhưng khác mask".

**Code nộp LeetCode (Cách 3):**

```python
class Solution:
    def minStickers(self, stickers: List[str], target: str) -> int:
        L = len(target)
        full = (1 << L) - 1
        counts = []
        for s in stickers:
            c = [0] * 26
            for ch in s:
                c[ord(ch) - 97] += 1
            counts.append(c)
        tidx = [ord(ch) - 97 for ch in target]
        INF = float('inf')
        dp = [INF] * (1 << L)
        dp[0] = 0
        for mask in range(full):
            if dp[mask] == INF:
                continue
            j = 0
            while mask >> j & 1:
                j += 1
            for cnt in counts:
                if cnt[tidx[j]] == 0:
                    continue
                left = cnt[:]
                new = mask
                for p in range(L):
                    if not new >> p & 1 and left[tidx[p]] > 0:
                        left[tidx[p]] -= 1
                        new |= 1 << p
                if dp[mask] + 1 < dp[new]:
                    dp[new] = dp[mask] + 1
        return -1 if dp[full] == INF else dp[full]
```

**Độ phức tạp.** `O(2^L · n · L)` thời gian, `O(2^L)` bộ nhớ. Với `L = 15`, `n = 50`: tệ nhất ~`2.4·10^7` bước — chấp nhận được ở C++/Java, hơi nặng với Python nhưng cắt tỉa chữ đầu giúp số nhãn thử mỗi mask thường nhỏ.

### 7.3. Cách 4 — BFS trên bitmask

**Giải thích.** Mỗi lần dùng một nhãn là một "bước" có chi phí 1 → BFS từ `mask = 0`; lần đầu chạm `full` là đáp án. Ưu điểm: dừng ngay khi tìm thấy, chỉ thăm các mask thực sự đạt được.

In [11]:
from collections import deque

def stickers_bfs(stickers, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    dist = {0: 0}
    q = deque([0])
    while q:
        mask = q.popleft()
        if mask == full:
            return dist[mask]
        j = 0
        while mask >> j & 1:
            j += 1
        c = ord(target[j]) - 97
        for cnt in counts:
            if cnt[c] == 0:
                continue
            new = apply_sticker(mask, cnt, target)
            if new not in dist:
                dist[new] = dist[mask] + 1
                q.append(new)
    return -1

print(stickers_bfs(st1, tg1), stickers_bfs(st2, tg2))

3 -1


**Code nộp LeetCode (Cách 4):**

```python
from collections import deque

class Solution:
    def minStickers(self, stickers: List[str], target: str) -> int:
        L = len(target); full = (1 << L) - 1
        tidx = [ord(ch) - 97 for ch in target]
        counts = []
        for s in stickers:
            c = [0] * 26
            for ch in s:
                c[ord(ch) - 97] += 1
            counts.append(c)
        dist = {0: 0}
        q = deque([0])
        while q:
            mask = q.popleft()
            if mask == full:
                return dist[mask]
            j = 0
            while mask >> j & 1:
                j += 1
            for cnt in counts:
                if cnt[tidx[j]] == 0:
                    continue
                left, new = cnt[:], mask
                for p in range(L):
                    if not new >> p & 1 and left[tidx[p]] > 0:
                        left[tidx[p]] -= 1
                        new |= 1 << p
                if new not in dist:
                    dist[new] = dist[mask] + 1
                    q.append(new)
        return -1
```

**Độ phức tạp.** Như Cách 3 trong trường hợp tệ nhất, `O(2^L · n · L)`; thường nhanh hơn vì dừng sớm.

## 8. TRACE từng bước

### 8.1. Bảng trạng thái BFS cho ví dụ 1 (bitmask đọc theo vị trí 0..5 của `"thehat"`)

Vị trí: `0:t 1:h 2:e 3:h 4:a 5:t`. Chuỗi bit đọc **từ trái sang phải là vị trí 0 → 5**.

| Lớp (số nhãn) | Mask | Đến từ | Vị trí chưa có đầu tiên (chữ) | Còn thiếu |
|---|---|---|---|---|
| 0 | `000000` | — | 0 (`t`) → chỉ thử `with` | t h e h a t |
| 1 | `110000` | `000000` + `with` (t→0, h→1) | 2 (`e`) → thử `example`, `science` | e h a t |
| 2 | `111010` | `110000` + `example` (e→2, a→4) | 3 (`h`) → thử `with` | h t |
| 2 | `111000` | `110000` + `science` (e→2) | 3 (`h`) | h a t |
| 3 | `111111` | `111010` + `with` (t→5, h→3) | — | ∅ → **đáp án 3** |

Hàm `trace_bfs` dưới đây in đúng các dòng này.

### 8.2. `trace_bfs`

In [12]:
def trace_bfs(stickers, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    show = lambda m: ''.join('1' if m >> j & 1 else '0' for j in range(L))
    missing = lambda m: ''.join(target[j] for j in range(L) if not m >> j & 1) or '∅'
    dist = {0: 0}
    q = deque([0])
    print(f"target={target}, stickers={stickers}")
    print(f"  lớp 0: {show(0)} (thiếu {missing(0)})")
    while q:
        mask = q.popleft()
        if mask == full:
            print(f"  => chạm full ở lớp {dist[mask]}")
            return dist[mask]
        j = 0
        while mask >> j & 1:
            j += 1
        for s, cnt in zip(stickers, counts):
            if cnt[ord(target[j]) - 97] == 0:
                continue
            new = apply_sticker(mask, cnt, target)
            if new not in dist:
                dist[new] = dist[mask] + 1
                q.append(new)
                print(f"  lớp {dist[new]}: {show(mask)} --{s}--> {show(new)} (thiếu {missing(new)})")
    print("  => không tới được full: -1")
    return -1

trace_bfs(st1, tg1)

target=thehat, stickers=['with', 'example', 'science']
  lớp 0: 000000 (thiếu thehat)
  lớp 1: 000000 --with--> 110000 (thiếu ehat)
  lớp 2: 110000 --example--> 111010 (thiếu ht)
  lớp 2: 110000 --science--> 111000 (thiếu hat)
  lớp 3: 111010 --with--> 111111 (thiếu ∅)
  lớp 3: 111000 --with--> 111101 (thiếu a)
  => chạm full ở lớp 3


3

Nhận xét: ở trạng thái đầu, vị trí chưa có đầu tiên là `0` (chữ `'t'`) → chỉ nhãn `with` được thử (vì `example`, `science` không có `'t'`). Đây chính là cắt tỉa chữ đầu đang hoạt động.

### 8.3. `trace_memo` — cây gọi đệ quy trên chuỗi còn thiếu

In [13]:
def trace_memo(stickers, target):
    counts = [to_count(s) for s in stickers]
    memo = {}
    def f(rem, depth):
        pad = '  ' * depth
        if not rem:
            print(f"{pad}f('') = 0")
            return 0
        if rem in memo:
            print(f"{pad}f('{rem}') = {memo[rem]} (đã nhớ)")
            return memo[rem]
        need = to_count(rem)
        first = ord(rem[0]) - 97
        best = float('inf')
        for s, cnt in zip(stickers, counts):
            if cnt[first] == 0:
                continue
            nxt = ''.join(chr(97 + i) * (need[i] - cnt[i]) for i in range(26) if need[i] > cnt[i])
            print(f"{pad}f('{rem}'): dùng '{s}' -> còn '{nxt}'")
            best = min(best, 1 + f(nxt, depth + 1))
        memo[rem] = best
        print(f"{pad}f('{rem}') = {best}")
        return best
    ans = f(''.join(sorted(target)), 0)
    return -1 if ans == float('inf') else ans

trace_memo(st1, tg1)

f('aehhtt'): dùng 'example' -> còn 'hhtt'
  f('hhtt'): dùng 'with' -> còn 'ht'
    f('ht'): dùng 'with' -> còn ''
      f('') = 0
    f('ht') = 1
  f('hhtt') = 2
f('aehhtt') = 3


3

## 9. Code nộp LeetCode

Chọn **Cách 2** (DFS + memo trên chuỗi còn thiếu, cắt tỉa chữ đầu) — ngắn, nhanh trên Python vì các trạng thái trùng chữ được gộp lại.

In [14]:
from typing import List
from functools import lru_cache

class Solution:
    def minStickers(self, stickers: List[str], target: str) -> int:
        counts = []
        for s in stickers:
            c = [0] * 26
            for ch in s:
                c[ord(ch) - 97] += 1
            counts.append(c)

        @lru_cache(maxsize=None)
        def f(rem):
            if not rem:
                return 0
            need = [0] * 26
            for ch in rem:
                need[ord(ch) - 97] += 1
            first = ord(rem[0]) - 97
            best = float('inf')
            for cnt in counts:
                if cnt[first] == 0:
                    continue
                nxt = ''.join(chr(97 + i) * (need[i] - cnt[i]) for i in range(26) if need[i] > cnt[i])
                best = min(best, 1 + f(nxt))
            return best

        ans = f(''.join(sorted(target)))
        return -1 if ans == float('inf') else ans

assert Solution().minStickers(["with", "example", "science"], "thehat") == 3
assert Solution().minStickers(["notice", "possible"], "basicbasic") == -1
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Công thức truy hồi.** Gọi `OPT(S)` là số nhãn ít nhất phủ multiset `S`. Nếu `S` rỗng thì 0. Ngược lại, xét một lời giải tối ưu (multiset nhãn `M`). Chữ `S[0]` được cung cấp bởi một nhãn `s ∈ M` chứa chữ đó. Bỏ `s` khỏi `M` → `M \ {s}` phủ được `S - s` (phần chữ `s` không cung cấp được). Nên `OPT(S) ≥ 1 + OPT(S - s)` với một `s` chứa `S[0]`. Ngược lại, với mọi `s` chứa `S[0]`, ta có `OPT(S) ≤ 1 + OPT(S - s)` (dùng `s` rồi lời giải tối ưu cho phần còn lại). Vậy:

> `OPT(S) = 1 + min_{s chứa S[0]} OPT(S - s)` — chính là điều Cách 2 tính.

**Kết thúc.** Mỗi bước `S - s` ngắn hơn `S` (vì `s` chứa `S[0]`), nên đệ quy dừng; memo đảm bảo mỗi trạng thái tính một lần.

**Không thể (−1).** Nếu không nhãn nào chứa một chữ của `target`, khi trạng thái có chữ đó đứng đầu thì không có nhánh nào → `inf` → `-1`. Nếu mọi chữ đều có trong một nhãn nào đó, luôn có lời giải (mỗi bước phủ ít nhất `S[0]`).

**Cách 3/4.** Trạng thái bitmask tương ứng 1-1 với multiset còn thiếu (theo quy tắc lấp "vị trí chưa có đầu tiên cùng chữ", các mask đạt được có dạng chuẩn hoá). Chuyển trạng thái giống Cách 2; DP theo `mask` tăng dần hợp lệ vì `new > mask`; BFS đúng vì mọi cạnh có trọng số 1.

## 11. Phân tích Time/Space

| Cách | Time | Space |
|---|---|---|
| 1. Brute force | `O(C(u+L-1, L) · 26)` | `O(u)` |
| 2. DFS + memo (chuỗi) | `O(#trạng thái · n · (26 + L))`, `#trạng thái ≤ 2^L` | `O(#trạng thái · L)` |
| 3. DP bitmask | `O(2^L · n · L)` | `O(2^L)` |
| 4. BFS bitmask | `O(2^L · n · L)` tệ nhất, dừng sớm | `O(2^L)` |

Đo thời gian trên một bộ lớn vừa phải (`L = 15`, 50 nhãn ngẫu nhiên):

In [15]:
import random, time, string

random.seed(691)
big_st = [''.join(random.choice(string.ascii_lowercase) for _ in range(random.randint(3, 10))) for _ in range(50)]
big_tg = ''.join(random.choice(string.ascii_lowercase[:12]) for _ in range(15))
print("target:", big_tg)
for name, fn in [("memo", stickers_memo), ("bitmask dp", stickers_bitmask_dp), ("bfs", stickers_bfs)]:
    t0 = time.perf_counter()
    r = fn(big_st, big_tg)
    print(f"{name:<11}: {r}  {time.perf_counter() - t0:.3f}s")

target: deaglgdbifgdcgf
memo       : 5  0.006s
bitmask dp : 5  0.007s
bfs        : 5  0.006s


## 12. Bộ test

1. Ví dụ đề.
2. Edge case: `target` một chữ; nhãn trùng hoàn toàn `target`; chữ lặp nhiều lần; không thể vì thiếu một chữ; nhãn vô ích.
3. Ngẫu nhiên có seed (bảng chữ nhỏ `a..e` để có nhiều trùng lặp): mọi cách so với brute force.

In [16]:
METHODS = [stickers_memo, stickers_bitmask_dp, stickers_bfs, lambda s, t: Solution().minStickers(s, t)]

def check(stickers, target, expected=None):
    ref = stickers_brute(stickers, target)
    for f in METHODS:
        got = f(stickers, target)
        assert got == ref, (stickers, target, got, ref)
    if expected is not None:
        assert ref == expected, (stickers, target, ref, expected)
    return ref

check(["with", "example", "science"], "thehat", 3)
check(["notice", "possible"], "basicbasic", -1)
check(["a"], "a", 1)
check(["abc"], "cba", 1)                       # nhãn trùng target (khác thứ tự)
check(["ab"], "aabb", 2)                       # chữ lặp: cần 2 bản
check(["a", "b"], "aaab", 4)
check(["xyz", "ab"], "abab", 2)                # nhãn vô ích
check(["abcd", "ace", "bdf"], "abcdef", 2)     # tham lam sẽ sai (xem Lỗi 1)
check(["these", "guess", "about", "garden", "him"], "atomher", 3)
check(["ab", "cd"], "abcde", -1)
print("Ví dụ + edge case: OK")

random.seed(2027)
for _ in range(400):
    n = random.randint(1, 4)
    stickers = [''.join(random.choice('abcde') for _ in range(random.randint(1, 4))) for _ in range(n)]
    target = ''.join(random.choice('abcde') for _ in range(random.randint(1, 7)))
    check(stickers, target)
print("400 bộ ngẫu nhiên: mọi cách khớp brute force")

Ví dụ + edge case: OK
400 bộ ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1: Tham lam — mỗi bước chọn nhãn phủ được nhiều chữ còn thiếu nhất

In [17]:
def stickers_greedy(stickers, target):
    need = Counter(target)
    used = 0
    while need:
        best = max(stickers, key=lambda s: sum((Counter(s) & need).values()))
        gain = Counter(best) & need
        if not gain:
            return -1
        need -= gain
        used += 1
    return used

ex = (["abcd", "ace", "bdf"], "abcdef")
print("tham lam:", stickers_greedy(*ex), "| đúng:", stickers_brute(*ex))

tham lam: 3 | đúng: 2


**Vì sao sai:** đây là bài phủ tập hợp, tham lam không tối ưu. `abcd` phủ 4 chữ nên được chọn trước, nhưng sau đó `e` và `f` nằm ở hai nhãn khác nhau → 3 nhãn; trong khi `ace + bdf` chỉ cần 2. Phải thử **mọi** nhãn (có cắt tỉa) như Cách 2.

### Lỗi 2: Coi nhãn như **tập** chữ (dùng một chữ cho mọi vị trí trùng)

In [18]:
def stickers_as_set(stickers, target):
    @lru_cache(maxsize=None)
    def f(rem):
        if not rem:
            return 0
        best = float('inf')
        for s in stickers:
            if rem[0] in s:
                nxt = ''.join(ch for ch in rem if ch not in s)     # SAI: xoá MỌI chữ trùng
                best = min(best, 1 + f(nxt))
        return best
    ans = f(''.join(sorted(target)))
    return -1 if ans == float('inf') else ans

print("sai:", stickers_as_set(["ab"], "aabb"), "| đúng:", stickers_brute(["ab"], "aabb"))

sai: 1 | đúng: 2


**Vì sao sai:** nhãn `"ab"` chỉ có **một** chữ `a` và **một** chữ `b`. Phải trừ theo **số lượng** (`need[i] - cnt[i]`), như `stickers_memo`.

### Lỗi 3: Quên trường hợp không thể → trả về `inf` (hoặc đệ quy vô hạn)

In [19]:
def stickers_no_minus1(stickers, target):
    counts = [to_count(s) for s in stickers]
    @lru_cache(maxsize=None)
    def f(rem):
        if not rem:
            return 0
        need = to_count(rem)
        best = float('inf')
        for cnt in counts:
            if cnt[ord(rem[0]) - 97]:
                nxt = ''.join(chr(97 + i) * (need[i] - cnt[i]) for i in range(26) if need[i] > cnt[i])
                best = min(best, 1 + f(nxt))
        return best
    return f(''.join(sorted(target)))               # SAI: không đổi inf thành -1

print("sai:", stickers_no_minus1(st2, tg2), "| đúng:", stickers_memo(st2, tg2))

sai: inf | đúng: -1


**Vì sao sai:** đề yêu cầu `-1`. Thêm `return -1 if ans == inf else ans`. Ngoài ra, nếu **không** cắt tỉa theo chữ đầu mà thử cả nhãn không giúp gì (`nxt == rem`), đệ quy sẽ gọi lại chính nó vô hạn — luôn đảm bảo mỗi bước phủ được ít nhất một chữ.

### Lỗi 4: Khoá memo là chuỗi **chưa sắp xếp**

Đúng kết quả nhưng `"eaht"` và `"heat"` thành hai khoá khác nhau → số trạng thái tăng vọt. Luôn chuẩn hoá trạng thái (sắp xếp, hoặc dùng tuple 26 số đếm).

## 14. Mẹo & nhận dạng pattern

- **Độ dài mục tiêu ≤ 15–20** → nghĩ tới **bitmask** trên vị trí.
- **"Ít nhất bao nhiêu món, mỗi món dùng vô hạn lần, để phủ đủ"** → DP/BFS trên trạng thái "còn thiếu gì" (giống *Coin Change* nhưng trạng thái là multiset).
- **Thứ tự không quan trọng** → cắt tỉa "phần tử đầu tiên chưa phủ phải được phủ ngay bởi lựa chọn này" — kỹ thuật cũng dùng trong *Smallest Sufficient Team* (LeetCode 1125), *Partition to K Equal Sum Subsets*.
- **Chuẩn hoá trạng thái** (sắp xếp chuỗi, lấp vị trí đầu tiên cùng chữ) để gộp các trạng thái tương đương.
- Có thể **loại nhãn bị trội**: nếu nhãn `A` có mọi chữ (tính số lượng, chỉ tính chữ có ích) ≥ nhãn `B` thì bỏ `B`.

## 15. Biến thể

1. **Mỗi loại nhãn chỉ có 1 bản** → thành bài chọn tập con (thêm chỉ số nhãn vào trạng thái hoặc DP 0/1).
2. **Nhãn có giá khác nhau, tối thiểu tổng tiền** → thay BFS bằng Dijkstra trên bitmask (hoặc DP với trọng số).
3. **In ra danh sách nhãn đã dùng** → lưu `parent` trong BFS rồi truy ngược.
4. **LeetCode 1125 — Smallest Sufficient Team**: phủ tập kỹ năng (tập, không lặp) bằng ít người nhất — DP bitmask trên kỹ năng.

Ví dụ biến thể 3 (in danh sách nhãn) và loại nhãn bị trội:

In [20]:
def stickers_with_plan(stickers, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    parent = {0: None}
    q = deque([0])
    while q:
        mask = q.popleft()
        if mask == full:
            plan = []
            while parent[mask] is not None:
                prev, s = parent[mask]
                plan.append(s)
                mask = prev
            return plan[::-1]
        j = 0
        while mask >> j & 1:
            j += 1
        for s, cnt in zip(stickers, counts):
            if cnt[ord(target[j]) - 97]:
                new = apply_sticker(mask, cnt, target)
                if new not in parent:
                    parent[new] = (mask, s)
                    q.append(new)
    return None

def remove_dominated(stickers, target):
    need = Counter(target)
    useful = [(s, Counter(s) & need) for s in stickers]
    keep = []
    for i, (s, ci) in enumerate(useful):
        if not ci:
            continue
        dominated = any(j != i and all(cj[ch] >= v for ch, v in ci.items()) and (cj != ci or j < i)
                        for j, (_, cj) in enumerate(useful))
        if not dominated:
            keep.append(s)
    return keep

print(stickers_with_plan(st1, tg1))
print(remove_dominated(["with", "example", "science", "ex", "wit"], "thehat"))
assert stickers_memo(remove_dominated(st1, tg1), tg1) == 3

['with', 'example', 'with']
['with', 'example']


## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | Thử multiset `t = 1, 2, …` nhãn | bùng nổ tổ hợp | `O(u)` | Trọng tài khi test, input rất nhỏ |
| 2. DFS + memo (chuỗi) | `f(rem) = 1 + min f(rem - s)`, nhãn chứa `rem[0]` | `O(#trạng thái · n · 26)` | `O(#trạng thái)` | Lời giải gọn, nhanh trên Python (khuyên dùng) |
| 3. DP bitmask | `dp[mask]` duyệt tăng dần, lấp vị trí đầu tiên | `O(2^L · n · L)` | `O(2^L)` | Khi muốn bottom-up, không đệ quy |
| 4. BFS bitmask | Mỗi nhãn = 1 bước, dừng khi chạm full | `O(2^L · n · L)` | `O(2^L)` | Khi cần truy vết kế hoạch, dừng sớm |

## 17. Bài tập

**Bài 1 (đoán output).** `minStickers(["ab", "bc"], "abc")`, `minStickers(["aaa"], "aaaa")`, `minStickers(["xy"], "z")`?

**Bài 2 (tính tay).** Vẽ cây gọi `f` (như 8.3) cho `stickers = ["ab", "bc", "ca"]`, `target = "abc"`.

**Bài 3 (điền TODO).** Hoàn thiện `subtract(rem, sticker)` trả về chuỗi còn thiếu (đã sắp xếp).

**Bài 4 (tự viết lại).** Viết lại `my_min_stickers(stickers, target)` bằng BFS trên bitmask.

**Bài 5 (mini challenge).** Mỗi nhãn có giá `cost[i]`; viết `min_cost_stickers(stickers, cost, target)` trả về tổng giá nhỏ nhất (Dijkstra trên bitmask), hoặc `-1`.

In [21]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def subtract(rem, sticker):
    # TODO: Counter(rem) - Counter(sticker), rồi ''.join(sorted(...elements()))
    pass

# Bài 4 — TODO
def my_min_stickers(stickers, target):
    # TODO: BFS từ mask 0; vị trí chưa có đầu tiên j; chỉ thử nhãn chứa target[j]; apply_sticker
    pass

# Bài 5 — TODO
def min_cost_stickers(stickers, cost, target):
    # TODO: heapq với (chi phí, mask); dist[mask]; chuyển như BFS nhưng cộng cost[i]
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: đếm chữ cần và chữ mỗi nhãn có.
- Mức 2: `"abc"`: một nhãn chỉ có 2 chữ → cần ≥ 2.
- Mức 3: `2`, `2` (mỗi nhãn chỉ 3 chữ `a`), `-1`.

**Bài 2.**
- Mức 1: `f("abc")`: chữ đầu `a` → thử `ab`, `ca`.
- Mức 2: `ab` → còn `"c"`; `ca` → còn `"b"`.
- Mức 3: `f("c") = 1` (dùng `bc` hoặc `ca`), `f("b") = 1` → đáp án `2`.

**Bài 3.**
- Mức 1: `Counter(rem) - Counter(sticker)` tự bỏ số âm.
- Mức 2: `.elements()` sinh lại các chữ theo số lượng.
- Mức 3: `''.join(sorted(...))`.

**Bài 4.**
- Mức 1: `dist = {0: 0}`, `deque([0])`.
- Mức 2: vòng `while mask >> j & 1: j += 1`.
- Mức 3: trả `dist[mask]` khi `mask == full`; hết hàng đợi → `-1`.

**Bài 5.**
- Mức 1: chi phí khác nhau → không dùng BFS thường được.
- Mức 2: `heapq.heappush(pq, (d + cost[i], new))`.
- Mức 3: bỏ qua phần tử lỗi thời `if d > dist[mask]: continue`.

## 19. Đáp án tham khảo

In [22]:
S = Solution()
# Bài 1
assert S.minStickers(["ab", "bc"], "abc") == 2
assert S.minStickers(["aaa"], "aaaa") == 2
assert S.minStickers(["xy"], "z") == -1

# Bài 2
assert S.minStickers(["ab", "bc", "ca"], "abc") == 2

# Bài 3
def subtract_ans(rem, sticker):
    return ''.join(sorted((Counter(rem) - Counter(sticker)).elements()))

assert subtract_ans("aehhtt", "with") == "aeht"
assert subtract_ans("aabb", "ab") == "ab"
print("Bài 1–3 OK")

Bài 1–3 OK


In [23]:
import heapq

# Bài 4
def my_min_stickers_ans(stickers, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    dist = {0: 0}
    q = deque([0])
    while q:
        mask = q.popleft()
        if mask == full:
            return dist[mask]
        j = 0
        while mask >> j & 1:
            j += 1
        for cnt in counts:
            if cnt[ord(target[j]) - 97]:
                new = apply_sticker(mask, cnt, target)
                if new not in dist:
                    dist[new] = dist[mask] + 1
                    q.append(new)
    return -1

# Bài 5
def min_cost_stickers_ans(stickers, cost, target):
    L = len(target)
    full = (1 << L) - 1
    counts = [to_count(s) for s in stickers]
    dist = {0: 0}
    pq = [(0, 0)]
    while pq:
        d, mask = heapq.heappop(pq)
        if d > dist[mask]:
            continue
        if mask == full:
            return d
        j = 0
        while mask >> j & 1:
            j += 1
        for cnt, c in zip(counts, cost):
            if cnt[ord(target[j]) - 97]:
                new = apply_sticker(mask, cnt, target)
                if d + c < dist.get(new, float('inf')):
                    dist[new] = d + c
                    heapq.heappush(pq, (d + c, new))
    return -1

def min_cost_brute(stickers, cost, target):
    need = Counter(target)
    best = float('inf')
    for t in range(1, len(target) + 1):
        for combo in combinations_with_replacement(range(len(stickers)), t):
            total = Counter()
            for i in combo:
                total += Counter(stickers[i])
            if all(total[ch] >= v for ch, v in need.items()):
                best = min(best, sum(cost[i] for i in combo))
    return -1 if best == float('inf') else best

assert min_cost_stickers_ans(["with", "example", "science"], [1, 1, 1], "thehat") == 3
assert min_cost_stickers_ans(["ab", "a", "b"], [10, 3, 3], "ab") == 6
random.seed(5)
for _ in range(200):
    n = random.randint(1, 3)
    stks = [''.join(random.choice('abcd') for _ in range(random.randint(1, 3))) for _ in range(n)]
    tg = ''.join(random.choice('abcd') for _ in range(random.randint(1, 5)))
    cost = [random.randint(1, 5) for _ in range(n)]
    assert my_min_stickers_ans(stks, tg) == stickers_brute(stks, tg)
    assert min_cost_stickers_ans(stks, cost, tg) == min_cost_brute(stks, cost, tg)
print("Bài 4–5 OK (đối chiếu brute force)")

Bài 4–5 OK (đối chiếu brute force)


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DFS + memo trên "phần còn thiếu":**

```python
@lru_cache(None)
def f(rem):                          # rem đã sắp xếp
    if not rem: return 0
    need = đếm(rem); best = inf
    for cnt in counts:
        if cnt[rem[0]] == 0: continue          # cắt tỉa chữ đầu
        nxt = ghép các chữ need[i] - cnt[i] > 0 theo thứ tự a..z
        best = min(best, 1 + f(nxt))
    return best
ans = f(sorted(target)); return -1 if ans == inf else ans
```

**Template BFS bitmask:** `mask` = vị trí đã có; `j` = bit 0 đầu tiên; thử nhãn chứa `target[j]`; áp nhãn = lấp vị trí chưa có đầu tiên cùng chữ.

**Checklist trước khi submit:**

- [ ] Trừ theo **số lượng** chữ, không theo tập.
- [ ] Chuẩn hoá trạng thái (chuỗi sắp xếp / bitmask theo quy tắc cố định).
- [ ] Cắt tỉa: chỉ thử nhãn chứa chữ đầu tiên còn thiếu (đồng thời tránh vòng lặp vô hạn).
- [ ] Trả `-1` khi không thể (đổi `inf` thành `-1`).
- [ ] Không dùng tham lam "phủ nhiều nhất".
- [ ] Test `["with","example","science"], "thehat" → 3`, `["notice","possible"], "basicbasic" → -1`, `["ab"], "aabb" → 2`.
- [ ] Tên method đúng: `minStickers(self, stickers, target)`.